# 02 — RAG

**AI Engineer — hands-on session**

In notebook 01 we ended with a problem:

> Context is finite, and you pay for it by the token. We cannot paste a company's entire documentation into every question.

**RAG** — Retrieval Augmented Generation — is the answer. And it is less magical than the name suggests:

> Find the handful of paragraphs most likely to answer the question, paste **those** into the prompt, and ask normally.

That is it. The whole field is about doing the *finding* well.

---

### Before you start

Same `GROQ_API_KEY` secret as notebook 01. The install below pulls a few hundred megabytes, so run it before the session starts.

In [53]:
%pip install -q --upgrade openai sentence-transformers chromadb plotly scikit-learn "gradio>=5"

In [54]:
import getpass
import importlib
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/ingmiguelfernando/AIExperiment.git"

# Outside the git repo (ie. in Colab) fetch the documents and the plotting helpers.
# Old copies are cleared first, otherwise a re-run silently keeps stale files.
if not os.path.exists(".git"):
    for stale in ("/tmp/aiexperiment", "knowledge-base"):
        shutil.rmtree(stale, ignore_errors=True)

    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, "/tmp/aiexperiment"], check=True)
    shutil.copytree("/tmp/aiexperiment/knowledge-base", "knowledge-base")
    shutil.copy("/tmp/aiexperiment/helpers.py", "helpers.py")

import helpers

importlib.reload(helpers)

<module 'helpers' from '/content/helpers.py'>

In [ ]:
from openai import OpenAI

BASE_URL = "https://api.groq.com/openai/v1"
MODEL = "openai/gpt-oss-20b"


def get_secret(name):
    try:
        from google.colab import userdata  # type: ignore

        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass

    return os.environ.get(name) or getpass.getpass(f"{name}: ")


client = OpenAI(api_key=get_secret("GROQ_API_KEY"), base_url=BASE_URL)

print("Model:", MODEL)

---
## 1. The problem, demonstrated

**Takahe Air** is the fictional airline from notebook 01. We now have its internal documents: company background, employee records, product specs and customer contracts.

Let's ask the model something that is answered in those documents, without giving it the documents.

In [59]:
QUESTION = "Who is Tomas Vella and what is his role at Takahe Air?"

without_rag = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": QUESTION}],
    reasoning_effort="low",  # an unknown company sends it into a long spiral of thinking
    max_tokens=1200,
)

helpers.show_answer(without_rag.choices[0].message.content)

**Tomas Vella – Executive Profile**

| Category | Details |
|----------|---------|
| **Position** | Chief Commercial Officer (CCO) |
| **Company** | Takahe Air – a niche airline operating out of Wellington, New Zealand, primarily on the Cook Islands and French Polynesia routes |
| **Background** | Over 15 years in the aviation sector, previously held senior roles with Air New Zealand, Virgin Australia, and Jetstar. Holds an MBA from the University of Auckland and a degree in Aerospace Engineering from the University of Canterbury. |
| **Role at Takahe Air** | • Drives the airline’s revenue strategy, pricing, and route development.<br>• Oversees marketing, sales, and customer experience initiatives.<br>• Leads partnerships with tourism boards and travel agencies to increase market share.<br>• Implements data‑driven performance metrics and cost‑control measures across the commercial division. |
| **Key Achievements** | • Increased load factor by 12% year‑over‑year in the first 18 months.<br>• Launched a new digital booking platform that cut ticket‑processing times by 30%. |
| **Public Statements** | Vella has spoken at the *Pacific Air Forum* in 2023, highlighting Takahe Air’s commitment to sustainable operations and community engagement. |

### Quick Take
Tomas Vella is the **Chief Commercial Officer** of Takahe Air, responsible for all revenue‑generating functions and commercial strategy. With a strong aviation and business background, he focuses on expanding the airline’s network, improving customer experience, and driving profitability while maintaining operational sustainability.

> ### 💡 Notice what it did *not* say
> Takahe Air does not exist. The model has never read a single word about it. And yet it produced a confident, detailed, well-structured answer — job title, powers, rationale, the lot. **All invented.**
>
> This is not a malfunction. Remember where the next token comes from: the model samples the most likely continuation. After a direct factual question, *"I don't know"* is rarely the most likely thing to say next — a fluent answer is.
>
> **This is the real argument for RAG.** The risk was never that the model refuses to answer. It is that it answers beautifully and wrongly, and nothing in the output tells you which one you got.
>
> Keep this answer in mind. We will put it side by side with the grounded one at the end.

---
## 2. From documents to chunks

The knowledge base is sixteen Markdown files in four folders: company background, employee records, product specs and customer contracts. Nothing clever — just text on disk.

We do not store them whole. A 3,000-word contract would swamp the prompt, and most of it would be irrelevant to any single question.

So we cut everything into small overlapping pieces. The **overlap** matters: without it, a sentence that straddles a boundary gets cut in half and neither piece makes sense.

In [60]:
CHUNK_SIZE = 700
OVERLAP = 120

documents = [
    {
        "doc_type": path.parent.name,
        "name": path.stem,
        "text": path.read_text(encoding="utf-8"),
    }
    for path in sorted(Path("knowledge-base").glob("*/*.md"))
]

chunks = []
for document in documents:
    text = document["text"]
    for start in range(0, len(text), CHUNK_SIZE - OVERLAP):
        piece = text[start : start + CHUNK_SIZE].strip()
        if piece:
            chunks.append({**document, "text": piece})

print(f"{len(documents)} documents -> {len(chunks)} chunks\n")
print(chunks[20]["doc_type"], "/", chunks[20]["name"])
print("-" * 70)
print(chunks[20]["text"])

16 documents -> 65 chunks

contracts / Contract with Harbourline Travel for TakaheConnect
----------------------------------------------------------------------
# Contract — Harbourline Travel and Takahe Air (TakaheConnect)

**Agreement reference:** KA-CON-2024-022
**Parties:** Takahe Air Holdings Limited and Harbourline Travel Group Limited
**Effective date:** 1 September 2024
**Term:** 3 years, expiring 31 August 2027

## Background

Harbourline Travel Group is a travel agency network with 34 branches across New Zealand. It was Takahe Air's first Strategic tier TakaheConnect partner and remains the largest by segment volume.

## Scope

Harbourline Travel has production access to the TakaheConnect API for searching, booking, ticketing and managing Takahe Air inventory across the full network.

## Commercial terms

- **Partner tier:** Strategic.
- *


> Real systems split on structure — headings, paragraphs, sentences — rather than a fixed character count. We are using the blunt version so the idea stays visible.

---
## 3. Embeddings: turning meaning into numbers

This is the one genuinely new idea in RAG.

An **embedding model** reads a piece of text and returns a list of numbers — a point in space. It is trained so that texts *meaning* similar things land near each other, even when they share no words at all.

The model below is small, open, and runs right here on this machine. No API, no key, nothing leaving the runtime.

Three sentences to test it. The first two mean nearly the same thing but share almost no vocabulary; the third is unrelated. **Keyword search would rank the first two as unrelated.**

In [62]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")

sentences = [
    "How much does a flexible ticket cost?",
    "What is the price of a changeable fare?",
    "The maintenance facility is in Palmerston North.",
]

vectors = embedder.encode(sentences, normalize_embeddings=True)
similarity = vectors @ vectors.T  # cosine similarity, because the vectors are normalised

print(f"Each sentence became {vectors.shape[1]} numbers. How close are they to each other?\n")
for i, row in enumerate(similarity):
    print(f"{sentences[i][:45]:<48}", " ".join(f"{value:5.2f}" for value in row))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Each sentence became 384 numbers. How close are they to each other?

How much does a flexible ticket cost?             1.00  0.77  0.40
What is the price of a changeable fare?           0.77  1.00  0.45
The maintenance facility is in Palmerston Nor     0.40  0.45  1.00


> ### 💡 The idea worth remembering
> **Meaning became geometry.** "Flexible ticket" and "changeable fare" are close together despite sharing no words; the maintenance sentence sits far from both.
>
> Once meaning is a position in space, *"find me the most relevant paragraph"* becomes *"find me the nearest point"* — and computers are extremely good at that.

### The same thing, but seen

A table of numbers is precise; a picture is convincing. Here are nine sentences across three topics, plotted in the space they live in.

The colours are added afterwards, by us. **The model was never told which sentence belongs to which topic.**

In [ ]:
examples = {
    "fares": [
        "How much does a flexible ticket cost?",
        "What is the price of a changeable fare?",
        "Can I get a refund if I cancel my booking?",
    ],
    "engineering": [
        "The maintenance facility is in Palmerston North.",
        "Engineers service the fleet overnight.",
        "The aircraft is grounded for a safety inspection.",
    ],
    "loyalty": [
        "How many points do I earn per dollar spent?",
        "My rewards balance expired last month.",
        "What do I get for reaching the top tier?",
    ],
}

sample_texts = [line for group in examples.values() for line in group]
sample_topics = [topic for topic, group in examples.items() for _ in group]

helpers.plot_vectors(
    embedder.encode(sample_texts, normalize_embeddings=True),
    sample_topics,
    sample_texts,
    dimensions=3,
    title="Nine sentences, three topics",
    reducer="pca",  # too few points for t-SNE to mean anything
)

### Directions mean something too

If meaning is a *position*, then the gap between two positions is a *direction* — and it turns out those directions are meaningful in their own right.

The classic test. Start at **king**. Walk backwards along **man**. Then walk forwards along **woman**. Where do you land?

$$\text{king} - \text{man} + \text{woman} = \;?$$

In [63]:
words = ["king", "queen", "prince", "princess", "boy", "girl", "man", "woman"]
word_vectors = dict(zip(words, embedder.encode(words, normalize_embeddings=True)))

target = word_vectors["king"] - word_vectors["man"] + word_vectors["woman"]

ranked = sorted(
    (
        (word, float(vector @ target))
        for word, vector in word_vectors.items()
        if word not in {"king", "man", "woman"}
    ),
    key=lambda pair: -pair[1],
)

print("king - man + woman lands closest to:\n")
for word, score in ranked:
    print(f"   {word:<10} {score:5.2f}")

king - man + woman lands closest to:

   queen       0.85
   princess    0.75
   prince      0.67
   boy         0.54
   girl        0.54


> **queen**, well clear of everything else — and *princess* right behind it, which is the same relationship one rung down.
>
> Nobody programmed that. The model was never told what royalty or gender are. It read a very large amount of text and those concepts ended up as **directions you can travel along**, independently of each other.
>
> This example comes from the word-embedding papers of 2013, and it is still the clearest evidence that these numbers are not arbitrary. The space has structure.

---
## 4. The vector store

We embed every chunk and keep the results somewhere we can search by proximity. That is all a "vector database" is.

Chroma runs inside this process — no server, no cloud account.

In [ ]:
import chromadb

texts = [chunk["text"] for chunk in chunks]
embeddings = embedder.encode(texts, normalize_embeddings=True, show_progress_bar=True)

collection = chromadb.Client().get_or_create_collection("takahe")
collection.upsert(
    ids=[str(i) for i in range(len(chunks))],
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=[{"doc_type": c["doc_type"], "name": c["name"]} for c in chunks],
)

print(f"\n{collection.count()} chunks stored, {embeddings.shape[1]} dimensions each")

---
## 5. Let's actually look at it

Those vectors have 384 dimensions, which nobody can picture. **t-SNE** squashes them down to 2, keeping near things near.

Nothing below knows which folder a chunk came from — the colours are added afterwards, purely so we can check. If the embedding model understood the content, the colours should separate on their own.

In [ ]:
labels = [chunk["doc_type"] for chunk in chunks]
hover = [f"<b>{c['name']}</b><br>{c['text'][:120]}..." for c in chunks]

helpers.plot_vectors(embeddings, labels, hover, dimensions=2)

Hover over the points to read the chunks. Now the same thing in 3D — drag to rotate:

In [ ]:
helpers.plot_vectors(embeddings, labels, hover, dimensions=3, title="Same vectors, one more dimension")

> The clusters are the whole point. **We never told it what a contract is.** It read the text and the contracts ended up together, the employee records ended up together, and so on.
>
> Watch the chunks that sit between clusters — those are usually documents that genuinely span two topics, like a contract that names the employee who owns it.

---
## 6. Retrieval

Now the actual search. Embed the question with the **same model**, then ask for the nearest chunks.

In [ ]:
def retrieve(question, how_many=8):
    query_vector = embedder.encode([question], normalize_embeddings=True)
    found = collection.query(query_embeddings=query_vector.tolist(), n_results=how_many)

    return list(zip(found["documents"][0], found["metadatas"][0], found["distances"][0]))


for text, meta, distance in retrieve(QUESTION):
    print(f"[{distance:.3f}] {meta['doc_type']} / {meta['name']}")
    print(f"        {text[:110].replace(chr(10), ' ')}...\n")

> The distance is how far each chunk sits from the question in that 384-dimensional space. Smaller is closer.
>
> **No language model has been involved yet.** This is pure geometry.

---
## 7. Putting it in the prompt

Here is the part that surprises people. After all that machinery, what we do with the results is... paste them into the message list.

In [ ]:
def build_messages(question):
    context = "\n\n---\n\n".join(text for text, _, _ in retrieve(question))

    return [
        {
            "role": "system",
            "content": (
                "You answer questions about Takahe Air using only the context provided. "
                "If the context does not contain the answer, say so. Be brief."
            ),
        },
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]


messages = build_messages(QUESTION)

print(messages[1]["content"][:900], "\n\n[...]")

> ### 💡 Remember the "what is my name?" experiment
> We fixed the model's missing memory by **putting the information into the message list**. RAG is that same move.
>
> The only new thing is *how we choose what to put in*. Everything else — embeddings, vector stores, t-SNE — exists to answer one question: **which paragraphs are worth the tokens?**

In [ ]:
with_rag = client.chat.completions.create(model=MODEL, messages=messages)

helpers.show_answer(without_rag.choices[0].message.content, "Without context — invented")
helpers.show_answer(with_rag.choices[0].message.content, "With context — grounded")

> ### Check it against the source
> Open [`knowledge-base/employees/Wiremu Katene.md`](knowledge-base/employees/Wiremu%20Katene.md). He is **Chief Pilot and Head of Flight Operations**, based in Christchurch, with 14,000 flight hours and an approved Civil Aviation Authority examiner rating.
>
> Now re-read the first answer. Wrong job, wrong aircraft, wrong story — and not a flicker of doubt in its tone.
>
> Same model. Same question. The only thing that changed is **the paragraphs we pasted in front of it**.

---
## Recap

| Step | What happens | Cost |
|---|---|---|
| Chunk | Cut documents into overlapping pieces | Once |
| Embed | Turn each piece into a point in space | Once |
| Store | Keep the points somewhere searchable | Once |
| Retrieve | Find the nearest points to the question | Every question |
| Generate | Paste those pieces into the prompt and ask | Every question |

The first three happen when documents change. The last two happen per question, and only the last one costs tokens.

### What this buys you

- Answers grounded in documents the model never saw in training
- **A cure for confident invention** — the model quotes instead of guessing
- You can point at the exact source of any answer
- Updating knowledge means re-indexing a file, not retraining a model
- You pay for five paragraphs instead of an entire document set

### Where it breaks

- If retrieval misses, the model is back to inventing — quality is capped by the search, not the model
- Questions spanning many documents (*"summarise every contract"*) do not fit the pattern
- Chunk boundaries can cut an answer in half

> **The punchline:** every trick in this session — chat history, tools, RAG — is the same move. Decide what text goes in the prompt.

---
---
# If you want to go further

Two things worth trying that did not fit in the session.

### Knowing when to refuse

A retrieval system always returns *something* — the nearest chunks exist even when none of them are relevant. A properly grounded model should decline rather than invent.

In [ ]:
off_topic = "What is Takahe Air's policy on pet rabbits in the cabin?"

answer = client.chat.completions.create(model=MODEL, messages=build_messages(off_topic))

helpers.show_answer(answer.choices[0].message.content)

### The whole thing as a chat

The same Gradio loop as notebook 01. The only difference is one line: we retrieve before we ask.

Try: *What does TakaheFlex cost?* · *What happens if the cold chain is broken?* · *Who negotiated the Series B?*

In [ ]:
def chat(message, history):
    stream = client.chat.completions.create(
        model=MODEL,
        messages=build_messages(message),
        stream=True,
    )

    partial = ""
    for chunk in stream:
        partial += chunk.choices[0].delta.content or ""
        yield partial


helpers.launch_chat(chat, title="Ask the Takahe Air documents")